# 00 — Build the derived dataset

**Data.** The ΔF/F traces and cell tables that this notebook reads (`data/`) are
not included in this repository. They will be made publicly available prior to
publication and are available from the authors on request. Every notebook is
provided with its executed outputs, so all figure panels and numbers can be
inspected without running the code.

Run this notebook once before any figure notebook. Starting from the per-cell
ΔF/F traces (`data/traces_dff_*.npy`) and the cell table (`data/cells.parquet`),
it produces everything the figure notebooks read:

| output | contents |
|---|---|
| `data/cell_qc.parquet` | activity criterion per cell (included / excluded), max ΔF/F |
| `data/episodes.parquet` | every active episode of every cell (`lib/event_detection.py`) |
| `data/cell_features.parquet` | per-cell features |

**ΔF/F** = (F − F₀)/F₀ with F₀ = the 12th percentile of the ROI's raw fluorescence
trace. Early (P2/P3) and intermediate (P6/P7) pupae were imaged at 0.806 s per
volume for 2 h.

In [1]:
import os
import sys
sys.path.insert(0, "../lib")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
plt.rcParams["pdf.fonttype"] = 42   # keep text editable in Illustrator
plt.rcParams["ps.fonttype"] = 42

from data_io import load_cells, load_traces, load_roi_coordinates
cells = load_cells()
FI = 0.806                    # s per volume
DATASETS = ["P2P3", "P6P7"]

from qc import compute_cell_qc, ACTIVITY_THRESH_DFF, ACTIVITY_MIN_CONSEC_FRAMES
from event_detection import detect_events
from features import extract_features, normalize_trace
print(f"{len(cells)} cells across {DATASETS}; frame interval {FI} s")

3470 cells across ['P2P3', 'P6P7']; frame interval 0.806 s


## Activity criterion

A cell is included if its ΔF/F₀ reaches at least 1 (a 100 % increase over baseline)
for at least 3 consecutive frames (~2.4 s) at any point in the recording. Cells
that never do are excluded from all further analyses.

In [2]:
qc_rows = []
for dataset in DATASETS:
    subset = cells[cells["dataset"] == dataset].sort_values("row_in_dataset")
    traces = load_traces(dataset)
    for cell_id, row_idx in zip(subset["cell_id"], subset["row_in_dataset"]):
        result = compute_cell_qc(traces[row_idx])
        result["cell_id"] = cell_id
        result["max_dff"] = float(traces[row_idx].max())
        qc_rows.append(result)
qc_df = pd.DataFrame(qc_rows)[["cell_id", "max_consec_above_1dff", "max_dff", "excluded", "exclusion_reason"]]
qc_df = cells[["cell_id", "dataset", "animal_idx", "ant", "plane", "roi_name"]].merge(qc_df, on="cell_id")
qc_df.to_parquet("../data/cell_qc.parquet", index=False)

summary = qc_df.groupby("dataset").agg(n_cells=("cell_id", "size"), n_excluded=("excluded", "sum"))
summary["pct_excluded"] = 100 * summary["n_excluded"] / summary["n_cells"]
summary

,n_cells,n_excluded,pct_excluded
dataset,,,
P2P3,1245,11,0.883534
P6P7,2225,162,7.280899


## Active episodes for every cell

`detect_events` finds **active episodes** (sustained excursions above baseline,
hysteresis rules on the 0.02 Hz-filtered trace) and **peaks** (local maxima with
prominence ≥ 3σ on the 0.05 Hz-filtered trace). The rules
and their rationale are documented in `lib/event_detection.py`. The episode table
is stored here (Figure S2 reads it); peak counts and intervals enter the feature
table below.

In [3]:
ep_rows = []
for dataset in DATASETS:
    sub = cells[(cells["dataset"] == dataset)].merge(qc_df[["cell_id", "excluded"]], on="cell_id")
    sub = sub[~sub["excluded"]].sort_values("row_in_dataset")
    traces = load_traces(dataset)
    for _, c in sub.iterrows():
        events, _ = detect_events(traces[c.row_in_dataset], FI)
        for e in events:
            ep_rows.append({"cell_id": c.cell_id, "dataset": dataset, "animal_idx": c.animal_idx, "ant": c.ant, **e})
    print(f"{dataset}: {len(sub)} included cells done", flush=True)
episodes = pd.DataFrame(ep_rows)
episodes.to_parquet("../data/episodes.parquet", index=False)
print(f"wrote data/episodes.parquet: {len(episodes)} episodes")

P2P3: 1234 included cells done


P6P7: 2063 included cells done


wrote data/episodes.parquet: 16607 episodes


\
## Per-cell features

`extract_features` (`lib/features.py`) summarizes each included cell: median
active-episode duration, fraction of the recording spent in an episode, number of
peaks, median inter-peak interval, autocorrelation decay time (τ), and the mean and
maximum ΔF/F. `mean_dff_normalized` is the mean of the cell's amplitude-normalized
trace, (ΔF/F − min)/(max − min).

In [4]:
\
rows = []
for dataset in DATASETS:
    subset = qc_df[(qc_df["dataset"] == dataset) & (~qc_df["excluded"])].merge(
        cells[["cell_id", "row_in_dataset"]], on="cell_id").sort_values("row_in_dataset")
    traces = load_traces(dataset)
    for _, r in subset.iterrows():
        trace = traces[r.row_in_dataset]
        feats = extract_features(trace, FI)
        feats["mean_dff_normalized"] = float(normalize_trace(trace).mean())
        feats["cell_id"] = r.cell_id
        rows.append(feats)
    print(f"{dataset}: {len(subset)} included cells", flush=True)
feat_df = cells[["cell_id", "dataset", "animal_idx", "ant", "plane", "roi_name"]].merge(pd.DataFrame(rows), on="cell_id")
feat_df.to_parquet("../data/cell_features.parquet", index=False)
feat_df.groupby("dataset")[["median_duration_s", "duty_cycle", "n_subpeaks", "median_interpeak_interval_s", "tau_s"]].median().T

P2P3: 1234 included cells


P6P7: 2063 included cells


dataset,P2P3,P6P7
median_duration_s,412.873500,818.090000
duty_cycle,0.498824,0.579908
n_subpeaks,14.000000,8.000000
median_interpeak_interval_s,308.496500,526.721000
tau_s,196.664000,291.772000
